# P-Question Pipeline — Colab Runner

In [ ]:
# ── Block 1: Git pull ─────────────────────────────────────────────────────────
import os

REPO = 'computational_semantics_course'

if os.path.isdir(f'/content/{REPO}'):
    %cd /content/{REPO}
    !git pull
else:
    %cd /content
    !git clone https://github.com/yuvalzohar12/computational_semantics_course.git
    %cd /content/{REPO}

# Clear stale bytecode so pulled changes take effect immediately
!find . -name '*.pyc' -delete 2>/dev/null
!find . -name '__pycache__' -type d -exec rm -rf {} + 2>/dev/null

!pip install -q -r requirements.txt
print('Done.')

In [ ]:
# ── Block 2: Data load ────────────────────────────────────────────────────────
from google.colab import drive
import os, shutil

drive.mount('/content/drive')

# ↓ Update this path if the dataset is stored elsewhere in your Drive
DRIVE_DATASET_PATH = '/content/drive/MyDrive/ConTRoL-dataset'

os.makedirs('data', exist_ok=True)
if os.path.exists(DRIVE_DATASET_PATH):
    shutil.copytree(DRIVE_DATASET_PATH, 'data/ConTRoL-dataset', dirs_exist_ok=True)
    print('Dataset copied successfully.')
else:
    print(f'Dataset not found at: {DRIVE_DATASET_PATH}')
    print('Update DRIVE_DATASET_PATH to the correct location in your Drive.')

In [ ]:
# ── Block 3: Model choice and API key ────────────────────────────────────────
import os

# ── Choose your model ────────────────────────────────────────────────────────
# Groq  (free tier, fast):  llama-3.1-8b-instant | llama-3.3-70b-versatile
# HF    (featherless-ai):   qwen2.5-32b-instruct
MODEL = 'llama-3.1-8b-instant'

# ── Paste your API key for the chosen provider ────────────────────────────────
os.environ['GROQ_API_KEY'] = 'YOUR_GROQ_API_KEY_HERE'   # ← Groq key
os.environ['HF_API_KEY']   = ''                          # ← HF key (qwen only)
os.environ['LAB_API_KEY']  = ''                          # ← lab/Tailscale (not reachable from Colab)

print(f'Model selected : {MODEL}')
print(f'GROQ key set   : {bool(os.environ["GROQ_API_KEY"].strip())}')
print(f'HF key set     : {bool(os.environ["HF_API_KEY"].strip())}')

In [ ]:
# ── Block 4: Run experiment ───────────────────────────────────────────────────

# Amount of data (samples to process)
# Set to None to run on the full dataset
LIMIT      = 10

# Max tokens the model may generate per LLM call
MAX_TOKENS = 2048

limit_flag = f'--limit {LIMIT}' if LIMIT else ''

# Patch DEFAULT_PARAMS with the chosen max_tokens before running
import sys, importlib
for mod in list(sys.modules.keys()):
    if any(k in mod for k in ('config', 'runner', 'prompts', 'open_source')):
        del sys.modules[mod]

import config.config as cfg
cfg.DEFAULT_PARAMS['max_tokens'] = MAX_TOKENS
print(f'max_tokens set to {MAX_TOKENS}')

!python main.py --experiment p_question --model {MODEL} {limit_flag}

In [ ]:
# ── Block 5: See results ──────────────────────────────────────────────────────
import json, glob

SEP = '=' * 72

result_files = sorted(glob.glob('results/p_question*.json'))
if not result_files:
    print('No results yet — run Block 4 first.')
else:
    with open(result_files[-1]) as f:
        data = json.load(f)

    samples = data['samples']
    total   = len(samples)
    correct = sum(1 for s in samples if s['label'] == s['prediction'])

    # ── Overall summary ───────────────────────────────────────────────────────
    print(SEP)
    print(f"File     : {result_files[-1]}")
    print(f"Model    : {data['metadata']['model']}")
    print(f"Accuracy : {correct}/{total} = {correct/total:.1%}" if total else 'No samples.')
    print()
    print(f"  {'Label':<15} {'Correct':>7} {'Total':>7} {'Accuracy':>9}")
    print(f"  {'-'*42}")
    for lbl in ['Entailment', 'Contradiction', 'Neutral']:
        gold = [s for s in samples if s['label'] == lbl]
        hits = sum(1 for s in gold if s['prediction'] == lbl)
        pct  = f'{hits/len(gold):.1%}' if gold else 'n/a'
        print(f"  {lbl:<15} {hits:>7} {len(gold):>7} {pct:>9}")

    # ── Per-sample detail ─────────────────────────────────────────────────────
    for s in samples:
        verdict = '✓' if s['label'] == s['prediction'] else '✗'

        print()
        print(SEP)
        print(f"[{verdict}] ID: {s['id']}   Gold: {s['label']}   Pred: {s['prediction']}")
        print(f"Premise    : {s['premise'][:200].strip()}...")
        print(f"Hypothesis : {s['hypothesis']}")

        # Stage 1a — all questions generated from the premise
        print()
        print(f"  Stage 1a — {len(s['stage1a_questions'])} questions generated")
        for i, q in enumerate(s['stage1a_questions'], 1):
            print(f"    {i:>2}. {q}")

        # Stage 1b — top-K with alignment scores
        print()
        print(f"  Stage 1b — top-{len(s['stage1b_aligned'])} by alignment score")
        print(f"    {'Score':>8}   Question")
        print(f"    {'-'*62}")
        for item in s['stage1b_aligned']:
            print(f"    {item['score']:>8.4f}   {item['question']}")

        # Stage 1c — evidence sentences + synthesised answer per question
        print()
        print(f"  Stage 1c — answers")
        for i, item in enumerate(s['stage1c_answers'], 1):
            status = 'UNANSWERABLE' if item['unanswerable'] else f"score={item['score']:.4f}"
            print(f"    Q{i}. [{status}] {item['question']}")
            evidence = item.get('evidence_sentences', [])
            if evidence:
                print(f"        Evidence ({len(evidence)} sentence{'s' if len(evidence) != 1 else ''}):")
                for ev in evidence:
                    print(f"          - {ev}")
            else:
                print(f"          Evidence : (none gathered)")
            print(f"        Answer   : {item['answer']}")

        # Stage 2 — final prediction
        print()
        print(f"  Stage 2 — classification")
        print(f"    Evidence : {s['stage2_evidence'] or '(empty — all unanswerable)'}")
        print(f"    Mode     : {s['stage2_mode']}   Metric: {s['alignment_metric']}")
        print(f"    Gold     : {s['label']}   Pred: {s['prediction']}   {verdict}")

        if s.get('warnings'):
            print(f"    Warnings : {'; '.join(s['warnings'])}")

    print()
    print(SEP)